# DECISION TREE

## What's in this notebook

- 1\. Gini Impurity
- 2\. Split Dataset
- 3\. Find Best Split
- 4\. Decision Tree
- 5\. Entropy and Information Gain
- 6\. Regression Trees
- 7\. Overfitting and Pruning
- 8\. Why Trees Don't Need Feature Scaling
- 9\. A Single Tree Is Unstable
- 10\. Feature Importance in a Tree
- 11\. Pros, Cons and Key Parameters
- 12\. Interview Questions
- 13\. In My Own Words

In [1]:
import numpy as np
from collections import Counter

## 1. Gini Impurity

In [2]:
def gini(y):
    counts = Counter(y)
    n = len(y)

    return 1 - sum((count / n) ** 2 for count in counts.values())

## 2. Split Dataset

In [3]:
def split(X, y, feature, threshold):

    left = X[:, feature] <= threshold
    right = X[:, feature] > threshold

    return (
        X[left], y[left],
        X[right], y[right]
    )

## 3. Find Best Split

In [4]:
def best_split(X, y):

    best_gain = -1
    best_feature = None
    best_threshold = None

    parent_gini = gini(y)

    for feature in range(X.shape[1]):

        for threshold in np.unique(X[:, feature]):

            X_left, y_left, X_right, y_right = split(
                X, y, feature, threshold
            )

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            n = len(y)

            weighted_gini = (
                len(y_left) / n * gini(y_left)
                +
                len(y_right) / n * gini(y_right)
            )

            gain = parent_gini - weighted_gini

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold

## 4. Decision Tree

In [5]:
class DecisionTree:

    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self.build_tree(X, y, depth=0)

    def build_tree(self, X, y, depth):

        # Stop if all samples belong to same class
        if len(set(y)) == 1:
            return Counter(y).most_common(1)[0][0]

        # Stop if maximum depth reached
        if depth == self.max_depth:
            return Counter(y).most_common(1)[0][0]

        feature, threshold = best_split(X, y)

        # No valid split
        if feature is None:
            return Counter(y).most_common(1)[0][0]

        X_left, y_left, X_right, y_right = split(
            X, y, feature, threshold
        )

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(
                X_left, y_left, depth + 1
            ),
            "right": self.build_tree(
                X_right, y_right, depth + 1
            )
        }

    def predict_one(self, x, node):

        # Leaf node
        if not isinstance(node, dict):
            return node

        if x[node["feature"]] <= node["threshold"]:
            return self.predict_one(x, node["left"])

        return self.predict_one(x, node["right"])

    def predict(self, X):

        return np.array([
            self.predict_one(x, self.tree)
            for x in X
        ])

In [6]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# creae random dataset
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Decision Tree

```text
1. Calculate Gini
        ↓
2. Try feature + threshold
        ↓
3. Calculate Gini after split
        ↓
4. Choose highest gain
        ↓
5. Recursively split
        ↓
6. Stop at max depth / pure node
        ↓
7. Leaf predicts majority class
```

In [7]:
tree = DecisionTree(max_depth=3)

tree.fit(X_train, y_train)

pred = tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.8


In [8]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

## 5. Entropy and Information Gain

Gini is one way to measure how mixed a node is. Entropy is the other common one:

```text
Gini      = 1 − Σ pᵢ²              0 = pure node, 0.5 = a 50/50 node (two classes)
Entropy   = −Σ pᵢ · log₂(pᵢ)        0 = pure node, 1.0 = a 50/50 node (two classes)

Information gain = impurity(parent) − Σ (n_child / n_parent) · impurity(child)
```

At every node the tree tries each feature and each threshold between sorted values, and keeps the split with the largest gain,
the biggest drop in impurity. This is **greedy**: it takes the best split now without looking ahead, so the final tree is good but not guaranteed to be the best possible tree.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier


def gini(p):
    p = np.asarray(p, dtype=float)
    return 1 - np.sum(p ** 2)


def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p))) + 0.0     # + 0.0 turns -0.0 into 0.0


def information_gain(parent, children, impurity):
    share = lambda counts: np.array(counts) / sum(counts)
    n = sum(sum(c) for c in children)
    return impurity(share(parent)) - sum(sum(c) / n * impurity(share(c)) for c in children)


for p in [[0.5, 0.5], [0.9, 0.1], [1.0, 0.0]]:
    print(f"class shares {p}: Gini {gini(p):.3f}, entropy {entropy(p):.3f}")

parent, children = [10, 10], [[8, 2], [2, 8]]
print(f"\nsplit 10/10 into 8/2 and 2/8: gain {information_gain(parent, children, entropy):.3f} with entropy, "
      f"{information_gain(parent, children, gini):.3f} with Gini")

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=42)
for criterion in ["gini", "entropy"]:
    score = cross_val_score(DecisionTreeClassifier(criterion=criterion, random_state=0), X_train, y_train, cv=5).mean()
    print(f"criterion={criterion}: 5-fold CV accuracy {score:.3f}")


class shares [0.5, 0.5]: Gini 0.500, entropy 1.000
class shares [0.9, 0.1]: Gini 0.180, entropy 0.469
class shares [1.0, 0.0]: Gini 0.000, entropy 0.000

split 10/10 into 8/2 and 2/8: gain 0.278 with entropy, 0.180 with Gini
criterion=gini: 5-fold CV accuracy 0.918
criterion=entropy: 5-fold CV accuracy 0.951


A 50/50 node scores 0.5 Gini and 1.0 entropy; a pure node scores 0 on both. Splitting 10/10 into 8/2 and 2/8 gains 0.278 with entropy and 0.180 with Gini.
On the breast cancer data entropy scored higher (0.951 vs 0.918), but with 426 training rows that gap is mostly noise: on most datasets the two pick nearly the same splits.

| | Gini | Entropy |
|---|---|---|
| Cost | no logarithm, slightly faster | log₂ for every candidate split, slightly slower |
| Behaviour | tends to split off the largest class | slightly more balanced splits |
| In practice | sklearn's default | results usually very close: try both with CV instead of assuming |

## 6. Regression Trees

For a numeric target the same algorithm uses a different impurity: the **variance (MSE)** of the target in a node.
A good split gives two children with less spread than the parent (variance reduction), and **each leaf predicts the mean** of its training rows.

So a regression tree's prediction is a step function: a tree of depth d has at most 2^d different outputs, and it **can't extrapolate**:
beyond the training range it keeps predicting the value of the last leaf.

In [2]:

from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 6, 200)).reshape(-1, 1)
y_reg = np.sin(x[:, 0]) + rng.normal(0, 0.1, 200)

reg = DecisionTreeRegressor(max_depth=2, random_state=0).fit(x, y_reg)
print("distinct predictions at depth 2:", np.unique(reg.predict(x)).round(3))
print("prediction at x = 6 and at x = 100 (far outside the data):", reg.predict([[6], [100]]).round(3))


distinct predictions at depth 2: [-0.739 -0.147  0.214  0.743]
prediction at x = 6 and at x = 100 (far outside the data): [-0.739 -0.739]


Depth 2 gives exactly 4 different predictions. At x = 100, far outside the 0–6 training range, it predicts -0.739, the same as at x = 6:
a tree only repeats leaf averages it has seen. Linear models and boosted models with linear leaves can extrapolate; plain trees and forests can't.

## 7. Overfitting and Pruning

An unlimited tree keeps splitting until every leaf is pure, so it memorizes the training data. Two ways to stop that:

- **Pre-pruning** (early stopping): limit the growth while building: `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_leaf_nodes`, `min_impurity_decrease`.
- **Post-pruning**: grow the full tree, then cut back branches that add little. sklearn uses **cost-complexity pruning**:
  it minimizes `error + α · (number of leaves)`, and a larger `ccp_alpha` removes more branches. Choose α with cross-validation.

In [3]:

full = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
alphas = full.cost_complexity_pruning_path(X_train, y_train).ccp_alphas[:-1]     # the last alpha prunes down to the root
cv_scores = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=0), X_train, y_train, cv=5).mean() for a in alphas]
best_alpha = alphas[int(np.argmax(cv_scores))]
print(f"{len(alphas)} candidate alphas, best by CV: {best_alpha:.4f}")

models = {
    "no limit": full,
    "pre-pruned (max_depth=3)": DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train),
    "post-pruned (best ccp_alpha)": DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=0).fit(X_train, y_train),
}
pd.DataFrame([{"model": name, "train": round(m.score(X_train, y_train), 3), "test": round(m.score(X_test, y_test), 3),
               "leaves": m.get_n_leaves(), "depth": m.get_depth()} for name, m in models.items()]).set_index("model")


12 candidate alphas, best by CV: 0.0063


,train,test,leaves,depth
model,,,,
no limit,1.000,0.937,18,7
pre-pruned (max_depth=3),0.977,0.944,7,3
post-pruned (best ccp_alpha),0.974,0.944,6,3


The unlimited tree scores 1.000 on training data but 0.937 on test, with 18 leaves. Both pruned trees score 0.944 on test
with only 7 leaves (max_depth=3) or 6 leaves (ccp_alpha = 0.0063): smaller, easier to read, and slightly better on new data.

| | Pre-pruning | Post-pruning (`ccp_alpha`) |
|---|---|---|
| How | stop growing early | grow the full tree, then cut back |
| Gain | fast, simple to tune | sees the whole tree first, so it can keep a weak split that leads to strong splits below it |
| Cost | can stop too early, for the same reason | slower: a full tree plus CV over many alphas |
| Pick it when | the default, and inside ensembles | a single tree that must stay small and readable |

## 8. Why Trees Don't Need Feature Scaling

A split asks "is this feature ≤ threshold?". Scaling a feature moves the thresholds but doesn't change the order of the rows,
so exactly the same rows go left and right. Trees also handle features in different units, and outliers in X, without trouble
(outliers in y still hurt regression trees, because leaves predict means).

In [4]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
raw = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
scaled = DecisionTreeClassifier(random_state=0).fit(scaler.transform(X_train), y_train)
print("same test predictions with and without scaling:", np.array_equal(raw.predict(X_test), scaled.predict(scaler.transform(X_test))))


same test predictions with and without scaling: True


The predictions are identical. That's why tree models (and random forests, XGBoost, LightGBM) skip the scaler, while KNN, SVM, linear models and neural networks need one.

## 9. A Single Tree Is Unstable

A small change in the training data can change the first split, and then everything below it changes too: a single tree has **high variance**.
Random forests average many trees trained on bootstrap samples with random feature subsets, which cancels most of this variance out (`2-random-forest.ipynb`).

In [5]:

rng = np.random.default_rng(0)
trees = []
for _ in range(3):
    rows = rng.choice(len(X_train), size=int(0.9 * len(X_train)), replace=False)       # a random 90% of the training rows
    trees.append(DecisionTreeClassifier(random_state=0).fit(X_train[rows], y_train[rows]))

print("first split:", [str(data.feature_names[t.tree_.feature[0]]) for t in trees])
print("leaves:", [int(t.get_n_leaves()) for t in trees])
print(f"test rows where tree 1 and tree 2 disagree: {(trees[0].predict(X_test) != trees[1].predict(X_test)).mean():.1%}")


first split: ['worst perimeter', 'mean concave points', 'worst area']
leaves: [16, 20, 17]
test rows where tree 1 and tree 2 disagree: 7.0%


Three trees, each trained on a different random 90% of the same data, start with three different first splits
(worst perimeter, mean concave points, worst area), end with 16, 20 and 17 leaves, and disagree on 7.0% of the test rows.

## 10. Feature Importance in a Tree

`feature_importances_` adds up how much each feature reduced impurity across all its splits (mean decrease in impurity), normalized to sum to 1.
Two known problems:
- it is computed on the training data and favours features with many distinct values (they offer more thresholds to split on)
- with correlated features (radius, perimeter and area here) the tree picks one and the others get almost nothing

Permutation importance on held-out data avoids the first problem (`13-ml-fundamentals/5-ml-explainability.ipynb`).

In [6]:

importance = pd.Series(raw.feature_importances_, index=data.feature_names).sort_values(ascending=False)
print(importance.head(3).round(3).to_string())
print(f"\nfeatures with importance > 0: {(importance > 0).sum()} of {len(importance)}")


worst radius            0.695
worst concave points    0.117
worst texture           0.054

features with importance > 0: 14 of 30


"worst radius" takes 0.695 of the total importance and only 14 of the 30 features are used at all. "worst perimeter" and "worst area"
are almost the same information as "worst radius", so the tree rarely needs them; that doesn't mean they are useless.

## 11. Pros, Cons and Key Parameters

| Good | Bad |
|---|---|
| easy to explain: a tree is a set of if-then rules | high variance: small data changes give a different tree |
| no scaling, handles mixed feature types and non-linear patterns | overfits without pruning |
| fast to train and to predict | can't extrapolate; regression output is a step function |
| sklearn ≥ 1.3 trees accept missing values natively | greedy splits are not globally optimal; axis-aligned splits need many steps for diagonal boundaries |

| Parameter | Default | What it does |
|---|---|---|
| `criterion` | `"gini"` | split quality: `"gini"`, `"entropy"`, `"log_loss"`; `"squared_error"` for regression |
| `max_depth` | None | maximum depth; the main pre-pruning control |
| `min_samples_split` | 2 | rows needed to split a node |
| `min_samples_leaf` | 1 | rows needed in each leaf; larger = smoother |
| `max_features` | None | features tried per split; random subsets are what random forests use |
| `ccp_alpha` | 0.0 | cost-complexity post-pruning strength |
| `class_weight` | None | `"balanced"` for imbalanced classes |

## 12. Interview Questions

### Q1. Explain a decision tree in 30 seconds.
A tree splits the data with yes/no questions on one feature at a time, choosing at each node the split that makes the children purest
(largest drop in Gini or entropy, or in variance for regression). Leaves predict the majority class or the mean. It is easy to explain and needs no scaling,
but it overfits and is unstable, so in practice it is pruned or used inside ensembles like random forests and gradient boosting.

### Q2. How does a tree choose a split?
For every feature it sorts the values, tries thresholds between neighbours, and computes the weighted impurity of the two children.
It keeps the split with the largest information gain. The search is greedy: best split now, no look-ahead.

### Q3. Gini vs entropy?
Both measure how mixed a node is (0 = pure). Gini = 1 − Σp² avoids the logarithm, so it is a bit faster; entropy = −Σp·log₂p gives slightly more balanced splits.
They usually produce almost the same tree, so Gini is the default and CV decides if it matters.

### Q4. How does a regression tree work, and can it extrapolate?
It splits to reduce the variance (MSE) of the target, and each leaf predicts the mean of its rows. The output is a step function, and outside the
training range it repeats the last leaf's value, so it can't extrapolate a trend.

### Q5. Why do trees overfit, and how do you stop it?
Without limits a tree splits until every leaf is pure, memorizing noise. Pre-prune (max_depth, min_samples_leaf, max_leaf_nodes),
post-prune with ccp_alpha chosen by CV, or use an ensemble.

### Q6. Pre-pruning vs post-pruning?
Pre-pruning stops growth early: fast, but it can stop before a weak split that would enable strong ones. Post-pruning grows the full tree
and cuts back the branches that add least (cost-complexity pruning): slower, but it decides with the whole tree in view.

### Q7. Why don't trees need feature scaling?
Splits compare one feature with a threshold. Scaling changes the threshold, not the order of the rows, so the same rows go to each side.

### Q8. Why is a single tree unstable, and how do ensembles fix it?
A small change in the data can change the top split and the whole tree below it (high variance). Random forests average many decorrelated trees
(bootstrap rows, random feature subsets) to cut variance; boosting adds small trees one after another to cut bias.

### Q9. What's wrong with impurity-based feature importance?
It is measured on training data, it favours features with many distinct values, and correlated features split the credit (or one takes it all).
Use permutation importance on validation data, or SHAP.

### Q10. How do trees handle categorical features and missing values?
sklearn trees need categories encoded (one-hot, or ordinal for ordered categories); LightGBM and CatBoost split on categories directly.
sklearn ≥ 1.3 handles missing values natively by learning which side missing rows should go; otherwise impute them first.

### Q11. When would you use a single tree instead of a random forest or boosting?
When the model must be fully explainable as rules (credit decisions, medical triage, a rule set for the business), on very small data,
or as a quick baseline. For the best accuracy on tabular data, gradient boosting usually wins.

## 13. In My Own Words

Write this from memory, without looking above:

- How a tree picks a split:
- Gini vs entropy, in one line:
- Pre-pruning vs post-pruning:
- Why a single tree is unstable, and what fixes it: